# Compresión del modelo cross (caídas)

Fase 2 — conversión y cuantización LiteRT del modelo `cross_final.keras` (CNN-BiLSTM, 2 canales AVM/GVM, ventana 3 s a 50 Hz).

## Objetivo

| Restricción | Valor objetivo |
| --- | --- |
| Tamaño del modelo de caídas | ~600 KB |
| Inferencia en CPU | <100 ms |
| Presupuesto colectivo (caídas + ECG + PPG) | <1 MB total, <300 ms combinado |

## Alcance

- Conversión TFLite fp32, float16, int8 dinámico e **int8 completo** (con dataset representativo de `set_b`).
- Verificación de paridad funcional contra el modelo Keras original y contra las métricas cross-dataset registradas en `cross_results.json`.
- Benchmark de tamaño y latencia en CPU (mediana y p95 sobre 100 corridas).
- **Capacidad de cómputo y viabilidad edge**: MACs y conteo de operaciones sobre el grafo TFLite, cross-check con el analizador de TF e inventario de ops frente a LiteRT Micro.

## 1 · Imports y constantes

In [1]:
import contextlib
import io
import json
import os
import pathlib
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

tf.get_logger().setLevel("ERROR")
tf.keras.config.disable_interactive_logging()

FS = 50
WINDOW_SEC = 3.0
WINDOW_SIZE = int(FS * WINDOW_SEC)   # 150
WINDOW_STEP = WINDOW_SIZE // 2       # 75
PEAK_MARGIN_SAMPLES = int(1.5 * FS)  # 75

CHANNEL_COLS = ["AVM", "GVM"]

ORO_DIR = pathlib.Path("../data/oro/falls")
MODELS_DIR = pathlib.Path("../data/modelos/falls")
TRAIN_PARQUET = ORO_DIR / "set_b.parquet"   # calibración int8: sin UMAFall
EXTERNAL_DATASET = "UMAFall"

SIZE_TARGET_KB = 600
LATENCY_TARGET_MS = 100
BUDGET_TOTAL_KB = 1024
BUDGET_ECG_KB = 90
BUDGET_PPG_KB = 412
N_LATENCY_RUNS = 100
INT8_CALIB_SAMPLES = 300

ESP32_REF_CLOCK_HZ = 240_000_000
ESP32_MACS_PER_CYCLE = (1, 4)
OP_OVERHEAD_US = (1, 5)

print(f"FS={FS} Hz | ventana={WINDOW_SIZE} | holdout={EXTERNAL_DATASET} | calibración int8={INT8_CALIB_SAMPLES} ventanas de {TRAIN_PARQUET.name}")


FS=50 Hz | ventana=150 | holdout=UMAFall | calibración int8=300 ventanas de set_b.parquet


## 2 · Holdout externo y utilidades (idénticas al entrenamiento)

In [2]:
def build_taxonomy_map():
    m = {}
    for f, codes in {
        "F1": ["F01","F02","F03"], "F2": ["F04"], "F3": ["F05"], "F4": ["F06","F07"],
        "F5": ["F10","F11","F12"], "F6": ["F08","F09"], "F7": ["F13","F14","F15"],
    }.items():
        for c in codes: m[("SisFall", c)] = f
    for f, codes in {
        "F1": ["F13","F14","F15"], "F2": ["F11"], "F3": ["F12"], "F4": ["F09","F10"],
        "F5": ["F01","F02","F03"], "F6": ["F04","F05"], "F7": ["F06","F07","F08"],
    }.items():
        for c in codes: m[("KFall", c)] = f
    for f, codes in {
        "F1": ["103","104","105","106","107","108","109","110","123","124","125","126"],
        "F2": ["101","102","121","122"],
        "F4": ["111","112","113","114","132","133","134","135"],
        "F5": ["115","116","117","118","119","120"],
        "F7": ["129","130","131"], "F8": ["127","128"],
    }.items():
        for c in codes: m[("FallAllD", c)] = f
    for f, codes in {"F5": ["5"], "F9": ["3","4"], "F10": ["1","2"]}.items():
        for c in codes: m[("UPFall", c)] = f
    for code in ["forwardFall", "backwardFall", "lateralFall"]:
        m[("UMAFall", code)] = "F9"
    return m

def map_to_unified_taxonomy(df):
    tax = build_taxonomy_map()
    out = df.copy()
    out["Fall_Type_Unified"] = [
        "ADL" if lbl == "ADL" else tax.get((ds, str(c)), "UNMAPPED")
        for ds, c, lbl in zip(out["Dataset"].values, out["Activity_Code"].values, out["Activity_Label"].values)
    ]
    return out

def create_windows(df):
    X_chunks, y_chunks = [], []
    for _, group in df.groupby(["Dataset", "Subject", "Activity_Code", "Trial"], sort=False):
        group = group.sort_values("Sample_Index").reset_index(drop=True)
        n = len(group)
        if n < WINDOW_SIZE:
            continue
        data = group[CHANNEL_COLS].values
        is_fall = group["Activity_Label"].iloc[0] == "Fall"
        n_w = (n - WINDOW_SIZE) // WINDOW_STEP + 1
        starts = np.arange(n_w) * WINDOW_STEP
        idx = starts[:, None] + np.arange(WINDOW_SIZE)[None, :]
        X_chunks.append(data[idx])
        if is_fall:
            peak_idx = int(np.argmax(group["AVM"].values))
            labels = (np.abs(starts + WINDOW_SIZE // 2 - peak_idx) <= PEAK_MARGIN_SAMPLES).astype(np.int32)
        else:
            labels = np.zeros(n_w, dtype=np.int32)
        y_chunks.append(labels)
    return np.concatenate(X_chunks, axis=0).astype(np.float32), np.concatenate(y_chunks)

def apply_scaler(X, scaler):
    n, t, c = X.shape
    return scaler.transform(X.reshape(-1, c)).reshape(n, t, c).astype(np.float32)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    return {
        "sensitivity": round(tp / (tp + fn), 4) if (tp + fn) else 0.0,
        "specificity": round(tn / (tn + fp), 4) if (tn + fp) else 0.0,
        "precision": round(tp / (tp + fp), 4) if (tp + fp) else 0.0,
        "tp": tp, "fn": fn, "tn": tn, "fp": fp,
    }

## 3 · Baseline Keras sobre el holdout externo

El holdout externo sigue el protocolo de evaluación cross-dataset de Santoyo-Ramón et al. (2022a) y Silva et al. (2024): un dataset nunca visto por el modelo (UMAFall) como prueba de generalización. Las métricas (sensibilidad, especificidad, precisión) se comparan contra las registradas en `cross_results.json`.


In [3]:
scaler = joblib.load(MODELS_DIR / "cross_scaler.joblib")
ext_df = pd.read_parquet(ORO_DIR / "set_a.parquet", filters=[("Dataset", "==", EXTERNAL_DATASET)])
ext_df = map_to_unified_taxonomy(ext_df)

X_ext, y_ext = create_windows(ext_df)
del ext_df
X_ext_s = apply_scaler(X_ext, scaler)
print(f"{EXTERNAL_DATASET}: {len(X_ext):,} ventanas, Fall={int(y_ext.sum()):,} ({y_ext.mean():.1%})")

model = tf.keras.models.load_model(MODELS_DIR / "cross_final.keras")
probs_keras = model.predict(X_ext_s, verbose=0).ravel()
baseline_metrics = compute_metrics(y_ext, (probs_keras >= 0.5).astype(int))
print("Keras fp32:", baseline_metrics)

with open(MODELS_DIR / "cross_results.json") as f:
    recorded = json.load(f)["external_final"]["natural"]
delta = {k: round(abs(baseline_metrics[k] - recorded[k]), 4) for k in ("sensitivity", "specificity", "precision")}
print("Registrado:", {k: recorded[k] for k in ("sensitivity", "specificity", "precision")})
print("Delta:", delta)
assert max(delta.values()) <= 0.01, f"Baseline Keras no reproduce cross_results.json: {delta}"

UMAFall: 4,315 ventanas, Fall=324 (7.5%)


Keras fp32: {'sensitivity': 0.9938, 'specificity': 0.9922, 'precision': 0.9122, 'tp': 322, 'fn': 2, 'tn': 3960, 'fp': 31}
Registrado: {'sensitivity': 0.9938, 'specificity': 0.9922, 'precision': 0.9122}
Delta: {'sensitivity': 0.0, 'specificity': 0.0, 'precision': 0.0}


## 4 · Reconstrucción con unroll=True y paridad

LiteRT recomienda convertir los RNN a operaciones LSTM fusionadas para máximo rendimiento (LiteRT, 2026a). Con Keras 3 ese camino falla porque el `Bidirectional(LSTM)` produce `tf.TensorListReserve` de forma dinámica; el workaround verificado es reconstruir la arquitectura con `unroll=True` y cargar los pesos. La arquitectura CNN-BiLSTM en edge está respaldada por FIBiLS (Gaud et al., 2025), que despliega un BiLSTM con poda y cuantización sobre ESP-8266.


In [4]:
def build_cnn_bilstm_unrolled():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(WINDOW_SIZE, len(CHANNEL_COLS))),
        tf.keras.layers.Conv1D(32, kernel_size=7, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv1D(64, kernel_size=5, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(64, kernel_size=3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, unroll=True)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])

model_compact = build_cnn_bilstm_unrolled()
model_compact.load_weights(MODELS_DIR / "cross_final.keras")

probs_unrolled = model_compact.predict(X_ext_s, verbose=0).ravel()
max_abs_diff = float(np.abs(probs_keras - probs_unrolled).max())
print(f"params={model_compact.count_params():,} | max|diff| Keras vs unroll = {max_abs_diff:.2e}")
assert max_abs_diff < 1e-6, "La reconstrucción unroll no reproduce al modelo original"

params=89,793 | max|diff| Keras vs unroll = 0.00e+00


## 5 · Conversión TFLite (fp32, float16, int8 dinámico e int8 completo)

Se agregan:

- **int8 dinámico**: pesos int8, activaciones fp32; sin calibración.
- **int8 completo**: pesos y activaciones int8 con dataset representativo; es el formato desplegable en microcontroladores LiteRT Micro (LiteRT, 2026c) y el que usa FIBiLS (Gaud et al., 2025) tras poda + cuantización.

La calibración usa ventanas de `set_b` (sin UMAFall) para no filtrar el holdout externo.


In [5]:
train_df = pd.read_parquet(TRAIN_PARQUET)
train_df = map_to_unified_taxonomy(train_df)
X_train, _ = create_windows(train_df)
del train_df

rng = np.random.default_rng(42)
calib_idx = rng.choice(len(X_train), size=INT8_CALIB_SAMPLES, replace=False)
X_calib = apply_scaler(X_train[calib_idx], scaler).astype(np.float32)
del X_train
print(f"Calibración int8: {len(X_calib):,} ventanas de {TRAIN_PARQUET.name}")

def representative_dataset():
    for i in range(len(X_calib)):
        yield [X_calib[i:i + 1]]


Calibración int8: 300 ventanas de set_b.parquet


In [ ]:
def convert(mode):
    converter = tf.lite.TFLiteConverter.from_keras_model(model_compact)
    if mode == "fp32":
        pass
    elif mode == "float16":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    elif mode == "int8_dynamic":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif mode == "int8_full":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8
    else:
        raise ValueError(mode)
    return converter.convert()

TFLITE_MODES = ("fp32", "float16", "int8_dynamic", "int8_full")
tflite_blobs = {mode: convert(mode) for mode in TFLITE_MODES}
for mode, blob in tflite_blobs.items():
    print(f"{mode}: {len(blob) / 1024:.1f} KB")


fp32: 581.5 KB
float16: 410.3 KB
int8_dynamic: 337.8 KB
int8_full: 507.1 KB


## 6 · Evaluación sobre el holdout y benchmark de latencia

Cada variante se evalúa sobre el holdout externo (4 315 ventanas) con el mismo umbral 0.5 del entrenamiento; las variantes int8 se cuantizan/decuantizan con los parámetros del propio modelo. Se reporta la latencia (mediana y p95 de 100 corridas, con warmup; protocolo propio) siguiendo la métrica de latencia de FIBiLS (Gaud et al., 2025) y Yu et al. (2020); aquí se mide en CPU de PC, no en la placa objetivo. La inferencia se ejecuta con el runtime LiteRT (`ai-edge-litert`); la conversión y el Model Analyzer siguen en `tf.lite`.


In [7]:
def _quantize_input(x, details):
    if details["dtype"] == np.int8:
        scale, zero = details["quantization"]
        return np.clip(np.round(x / scale) + zero, -128, 127).astype(np.int8)
    return x.astype(np.float32)

def _dequantize_output(y, details):
    if details["dtype"] == np.int8:
        scale, zero = details["quantization"]
        return (float(y) - zero) * scale
    return float(y)

def predict_tflite(blob, X):
    interp = Interpreter(model_content=blob)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    probs = np.empty(len(X), dtype=np.float32)
    for i in range(len(X)):
        interp.set_tensor(inp["index"], _quantize_input(X[i:i + 1], inp))
        interp.invoke()
        probs[i] = _dequantize_output(interp.get_tensor(out["index"]).ravel()[0], out)
    return probs

def benchmark_tflite(blob, x, runs=N_LATENCY_RUNS):
    interp = Interpreter(model_content=blob)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    xq = _quantize_input(x, inp)
    interp.set_tensor(inp["index"], xq)
    for _ in range(5):
        interp.invoke()
    times = []
    for _ in range(runs):
        interp.set_tensor(inp["index"], xq)
        t0 = time.perf_counter()
        interp.invoke()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times)), float(np.percentile(times, 95))

x_bench = X_ext_s[:1].copy()
results = {}
for mode, blob in tflite_blobs.items():
    probs = predict_tflite(blob, X_ext_s)
    metrics = compute_metrics(y_ext, (probs >= 0.5).astype(int))
    median_ms, p95_ms = benchmark_tflite(blob, x_bench)
    results[mode] = {
        "size_kb": round(len(blob) / 1024, 1),
        "latency_ms_median": round(median_ms, 3),
        "latency_ms_p95": round(p95_ms, 3),
        "max_abs_diff_vs_keras": float(np.abs(probs_keras - probs).max()),
        "metrics": metrics,
    }
    print(mode, results[mode])


fp32 {'size_kb': 581.5, 'latency_ms_median': 0.986, 'latency_ms_p95': 4.011, 'max_abs_diff_vs_keras': 8.344650268554688e-07, 'metrics': {'sensitivity': 0.9938, 'specificity': 0.9922, 'precision': 0.9122, 'tp': 322, 'fn': 2, 'tn': 3960, 'fp': 31}}


float16 {'size_kb': 410.3, 'latency_ms_median': 1.04, 'latency_ms_p95': 6.504, 'max_abs_diff_vs_keras': 0.0011491775512695312, 'metrics': {'sensitivity': 0.9938, 'specificity': 0.9922, 'precision': 0.9122, 'tp': 322, 'fn': 2, 'tn': 3960, 'fp': 31}}


int8_dynamic {'size_kb': 337.8, 'latency_ms_median': 0.365, 'latency_ms_p95': 0.765, 'max_abs_diff_vs_keras': 0.02709829807281494, 'metrics': {'sensitivity': 0.9938, 'specificity': 0.992, 'precision': 0.9096, 'tp': 322, 'fn': 2, 'tn': 3959, 'fp': 32}}


int8_full {'size_kb': 507.1, 'latency_ms_median': 0.404, 'latency_ms_p95': 0.889, 'max_abs_diff_vs_keras': 0.7137656211853027, 'metrics': {'sensitivity': 0.9907, 'specificity': 0.9905, 'precision': 0.8942, 'tp': 321, 'fn': 3, 'tn': 3953, 'fp': 38}}


## 7 · Capacidad de cómputo y viabilidad edge (MACs + ESP32)

La revisión de la literatura muestra que no es común evaluar **FLOPs/MACs**: el estándar para costo de despliegue es latencia + RAM pico + flash sobre el MCU (Gaud et al., 2025), complementado con tamaño de modelo (Villa & Casilari, 2026; Guo & Nakayama, 2025) y número de parámetros (Ordóñez & Roggen, 2016).

Se adopta entonces:

1. **Walker sobre el grafo TFLite**: MACs de capas convolucionales/densas y conteo de operaciones por tipo (mide el artefacto comprimido; independiente del hardware). La cuantización no cambia los MACs, solo el costo por operación.
2. **Cross-check** con el analizador de TF (`tf.compat.v1.profiler`, API heredada aún incluida en TF 2.21) sobre el modelo Keras equivalente, y con el Model Analyzer de LiteRT (LiteRT, 2026b) para estructura del grafo y buffers de memoria.
3. **Estimación ESP32** con supuestos explícitos y ajustables (reloj 240 MHz de referencia; 1–4 MACs/ciclo; 1–5 µs de overhead por op).


In [8]:
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

def tflite_op_stats(blob):
    interp = Interpreter(model_content=blob)
    interp.allocate_tensors()
    ops = interp._get_ops_details()
    shapes = {t["index"]: tuple(int(d) for d in t["shape"]) for t in interp.get_tensor_details()}
    counts, macs = {}, 0
    for o in ops:
        name = o["op_name"]
        counts[name] = counts.get(name, 0) + 1
        ins = [shapes[i] for i in o["inputs"] if i >= 0]
        outs = [shapes[i] for i in o["outputs"] if i >= 0]
        if not ins or not outs:
            continue
        if name == "CONV_2D":
            macs += int(np.prod(outs[0])) * ins[1][-3] * ins[1][-2] * ins[0][-1]
        elif name == "DEPTHWISE_CONV_2D":
            macs += int(np.prod(outs[0])) * ins[1][-3] * ins[1][-2]
        elif name == "FULLY_CONNECTED":
            macs += int(np.prod(outs[0])) * ins[1][-1]
    return counts, macs

op_stats = {mode: tflite_op_stats(blob) for mode, blob in tflite_blobs.items()}
macs_by_variant = {mode: s[1] for mode, s in op_stats.items()}
assert len(set(macs_by_variant.values())) == 1, macs_by_variant
ops_by_type = op_stats["int8_full"][0]
macs_total = macs_by_variant["int8_full"]
ops_total = sum(ops_by_type.values())
print(f"MACs por variante: {macs_by_variant}")
print(f"Ops totales: {ops_total} | por tipo: {dict(sorted(ops_by_type.items(), key=lambda kv: -kv[1]))}")

def frozen_forward(x):
    return model_compact(x)

frozen = convert_variables_to_constants_v2(
    tf.function(frozen_forward).get_concrete_function(
        tf.TensorSpec([1, WINDOW_SIZE, len(CHANNEL_COLS)], tf.float32)
    )
)
def proto_float_ops(node):
    return node.float_ops + sum(proto_float_ops(child) for child in node.children)

tfprof_options = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
tfprof_options["output"] = "none"
report = tf.compat.v1.profiler.profile(frozen.graph, options=tfprof_options)
tfprof_flops = float(proto_float_ops(report))
print(f"Cross-check tfprof (Keras unroll): {tfprof_flops / 1e6:.2f} MFLOPs ≈ {tfprof_flops / 2e6:.2f} MMACs")
ratio = abs(macs_total - tfprof_flops / 2) / (tfprof_flops / 2)
assert ratio < 0.05, (macs_total, tfprof_flops)
print(f"Diferencia walker vs tfprof/2: {ratio:.1%} (tfprof suma otros ops float)")

def analyzer_summary(blob):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        tf.lite.experimental.Analyzer.analyze(model_content=blob)
    text = buf.getvalue()
    summary = {"graph_lines": len(text.splitlines())}
    for key, pat in {
        "model_bytes": r"Model size:\s+(\d+) bytes",
        "non_data_bytes": r"Non-data buffer size:\s+(\d+) bytes",
        "data_bytes": r"Total data buffer size:\s+(\d+) bytes",
    }.items():
        hit = re.search(pat, text)
        summary[key] = int(hit.group(1)) if hit else None
    return summary

buffers_by_variant = {mode: analyzer_summary(tflite_blobs[mode]) for mode in ("float16", "int8_full")}
for mode, summary in buffers_by_variant.items():
    print(f"Analyzer {mode}: {summary}")

latency_estimates = {}
for mpc in ESP32_MACS_PER_CYCLE:
    for overhead_us in OP_OVERHEAD_US:
        cycles = macs_total / mpc + ops_total * overhead_us * 1e-6 * ESP32_REF_CLOCK_HZ
        latency_estimates[f"{mpc}mac_ciclo_{overhead_us}us_op"] = round(cycles / ESP32_REF_CLOCK_HZ * 1000, 2)
print(f"Estimación ESP32 @ {ESP32_REF_CLOCK_HZ / 1e6:.0f} MHz (ms): {latency_estimates}")


MACs por variante: {'fp32': 4949760, 'float16': 4949760, 'int8_dynamic': 4949760, 'int8_full': 4949760}
Ops totales: 1068 | por tipo: {'LOGISTIC': 223, 'ADD': 222, 'MUL': 222, 'FULLY_CONNECTED': 149, 'TANH': 148, 'SPLIT': 74, 'DELEGATE': 8, 'EXPAND_DIMS': 5, 'RESHAPE': 5, 'CONV_2D': 3, 'MAX_POOL_2D': 2, 'SHAPE': 1, 'STRIDED_SLICE': 1, 'PACK': 1, 'FILL': 1, 'TRANSPOSE': 1, 'UNPACK': 1, 'CONCATENATION': 1}


Cross-check tfprof (Keras unroll): 10.01 MFLOPs ≈ 5.01 MMACs
Diferencia walker vs tfprof/2: 1.1% (tfprof suma otros ops float)
Analyzer float16: {'graph_lines': 2465, 'model_bytes': 420124, 'non_data_bytes': 241297, 'data_bytes': 178827}
Analyzer int8_full: {'graph_lines': 2435, 'model_bytes': 519256, 'non_data_bytes': 429271, 'data_bytes': 89985}
Estimación ESP32 @ 240 MHz (ms): {'1mac_ciclo_1us_op': 21.69, '1mac_ciclo_5us_op': 25.96, '4mac_ciclo_1us_op': 6.22, '4mac_ciclo_5us_op': 10.5}


## 8 · Selección, presupuesto colectivo y persistencia

El entregable es `cross_float16.tflite`. Se agrega `cross_int8_full.tflite` como alternativa de despliegue en microcontrolador (LiteRT Micro; Gaud et al., 2025). Se verifica el presupuesto colectivo de los tres módulos (<1 MB) con las estimaciones del plan (ECG 90 KB, SpO2 412 KB).


In [9]:
deliverable = "float16"
edge_candidate = "int8_full"
fp16 = results[deliverable]
total_kb = fp16["size_kb"] + BUDGET_ECG_KB + BUDGET_PPG_KB

assert fp16["size_kb"] <= SIZE_TARGET_KB, fp16
assert fp16["latency_ms_median"] < LATENCY_TARGET_MS, fp16
assert max(abs(fp16["metrics"][k] - baseline_metrics[k]) for k in ("sensitivity", "specificity")) <= 0.01, fp16
assert total_kb < BUDGET_TOTAL_KB, total_kb

for mode, blob in tflite_blobs.items():
    (MODELS_DIR / f"cross_{mode}.tflite").write_bytes(blob)

report = {
    "source_model": "cross_final.keras",
    "baseline_keras": baseline_metrics,
    "recorded_cross_results": {k: recorded[k] for k in ("sensitivity", "specificity", "precision")},
    "unroll_max_abs_diff": max_abs_diff,
    "variants": results,
    "deliverable": f"cross_{deliverable}.tflite",
    "edge_candidate": f"cross_{edge_candidate}.tflite",
    "collective_budget": {
        "falls_kb": fp16["size_kb"], "ecg_kb": BUDGET_ECG_KB, "ppg_kb": BUDGET_PPG_KB,
        "total_kb": round(total_kb, 1), "limit_kb": BUDGET_TOTAL_KB,
    },
    "compute": {
        "macs_total": macs_total,
        "macs_by_variant": macs_by_variant,
        "macs_invariant": True,
        "ops_total": ops_total,
        "ops_by_type": ops_by_type,
        "tfprof_flops": tfprof_flops,
        "tflite_buffers": buffers_by_variant,
    },
    "edge_feasibility": {
        "board_undecided": True,
        "esp32_ref_clock_hz": ESP32_REF_CLOCK_HZ,
        "macs_per_cycle_assumed": list(ESP32_MACS_PER_CYCLE),
        "op_overhead_us_assumed": list(OP_OVERHEAD_US),
        "latency_ms_estimate": latency_estimates,
        "arena_measured": False,
        "fase3_protocol": "latencia + RAM pico + flash en placa (Gaud et al., 2025)",
    },
}
with open(MODELS_DIR / "cross_compression_results.json", "w") as f:
    json.dump(report, f, indent=2)

for name in ("cross_fp32.tflite", "cross_float16.tflite", "cross_int8_dynamic.tflite",
             "cross_int8_full.tflite", "cross_compression_results.json"):
    path = MODELS_DIR / name
    print(f"OK {path} ({path.stat().st_size / 1024:.1f} KB)")


OK ..\data\modelos\falls\cross_fp32.tflite (581.5 KB)
OK ..\data\modelos\falls\cross_float16.tflite (410.3 KB)
OK ..\data\modelos\falls\cross_int8_dynamic.tflite (337.8 KB)
OK ..\data\modelos\falls\cross_int8_full.tflite (507.1 KB)
OK ..\data\modelos\falls\cross_compression_results.json (3.6 KB)


## Conclusión

La conversión TFLite del modelo cross se validó en cuatro variantes (fp32, float16, int8 dinámico e int8 completo) con verificación de paridad funcional (salidas Keras vs TFLite) y de métricas contra el holdout externo UMAFall registrado en `cross_results.json`.

El entregable de FASE 2 es `cross_float16.tflite`: cumple el objetivo de tamaño (~600 KB) y latencia (<100 ms) con margen, y mantiene el presupuesto colectivo de los tres módulos por debajo de 1 MB. El candidato edge es `cross_int8_full.tflite` (pesos y activaciones int8, apto para LiteRT Micro).

La capacidad de cómputo se reporta como MACs y conteo de operaciones sobre el grafo TFLite (invariante entre variantes de cuantización: la cuantización reduce bytes, no operaciones), con cross-check independiente del analizador de TF y una estimación paramétrica de latencia ESP32. Siguiendo a FIBiLS (Gaud et al., 2025), la validación definitiva en el dispositivo — latencia real, RAM pico y flash — corresponde a FASE 3; la placa sigue sin decidirse y las suposiciones de reloj/MACs por ciclo quedan persistidas en el JSON.

**Límites:** latencia medida en CPU de PC; sin estimación de arena SRAM (diferida a FASE 3); el grafo desenrollado (~1060 ops primitivas (más pseudo-ops del delegate del runtime)) tiene mayor overhead de intérprete que un LSTM fusionado, que Keras 3 no permite generar (LiteRT, 2026a).


## Referencias

- Gaud, N., Rathore, M., Suman, U., & Semwal, V. B. (2025). FIBiLS: Fall detection of healthy elderly using IMU sensor and BiLSTM model. *IEEE Sensors Journal, 25*(19), 37124–37131. https://doi.org/10.1109/JSEN.2025.3602945
- Guo, P., & Nakayama, M. (2025). A feature engineering method for smartphone-based fall detection. *Sensors, 25*(20), Article 6500. https://doi.org/10.3390/s25206500
- LiteRT. (2026a). *Convert TensorFlow RNN models to LiteRT* [Documentación]. Google AI Edge. https://developers.google.com/edge/litert/conversion/tensorflow/rnn
- LiteRT. (2026b). *TensorFlow Lite Model Analyzer* [Documentación]. Google AI Edge. https://ai.google.dev/edge/api/tflite/python/tf/lite/experimental/Analyzer
- LiteRT. (2026c). *Post-training quantization* [Documentación]. Google AI Edge. https://ai.google.dev/edge/litert/models/post_training_quantization
- Ordóñez, F. J., & Roggen, D. (2016). Deep convolutional and LSTM recurrent neural networks for multimodal wearable activity recognition. *Sensors, 16*(1), Article 115. https://doi.org/10.3390/s16010115
- Santoyo-Ramón, J. A., Casilari, E., & Cano-García, J. M. (2022a). A cross-dataset evaluation of wearable fall detection systems. En *PETRA '22: Proceedings of the 15th International Conference on PErvasive Technologies Related to Assistive Environments* (pp. 1–6). Association for Computing Machinery. https://doi.org/10.1145/3529190.3529191
- Silva, C. A., Casilari, E., & García-Bermúdez, R. (2024). Cross-dataset evaluation of wearable fall detection systems using data from real falls and long-term monitoring of daily life. *Measurement, 235*, Article 114992. https://doi.org/10.1016/j.measurement.2024.114992
- Villa, M., & Casilari, E. (2026). The impact of the accelerometer sampling rate on the performance of machine and deep learning models in wearable fall-detection systems. *Sensors, 26*(1), Article 162. https://doi.org/10.3390/s26010162
- Yu, X., Qiu, H., & Xiong, S. (2020). A novel hybrid deep neural network to predict pre-impact fall for older people based on wearable inertial sensors. *Frontiers in Bioengineering and Biotechnology, 8*, Article 63. https://doi.org/10.3389/fbioe.2020.00063
